# კვირა II: მონაცემთა მომზადება და მოდელის შეფასება (Data Preprocessing & Model Evaluation)
**კურსი:** ხელოვნური ინტელექტი და მანქანური სწავლების საწყისები (BTU)  
**სახელმძღვანელო:** Chris Albon, *Machine Learning with Python Cookbook* (1st Edition, O'Reilly)  

---
### 🎯 ლექციის დაპირება (The Promise):
> *ამ Notebook-ის დასრულებისას თქვენს ხელთ იქნება Scikit-Learn-ის გამართული, მონაცემთა გაჟონვისგან დაცული Pipeline. თქვენ რეალურ სამედიცინო მონაცემებზე გაასუფთავებთ და დაასტანდარტებთ მახასიათებლებს, k-fold კროს-ვალიდაციით იპოვით საუკეთესო k-ს, ააგებთ აღრევის მატრიცას (Confusion Matrix) და გამოთვლით Precision-ს, Recall-სა და ROC-AUC-ს — და შეძლებთ ახსნათ, რატომ არის Accuracy მატყუარა და როგორ გადაარჩენს სწორად შერჩეული გადაწყვეტილების ზღურბლი (Threshold) ადამიანის სიცოცხლეს.*

### 📚 ლაბორატორიის სტრუქტურა:
1. **მასშტაბირების კატასტროფა:** რატომ კლავს სხვადასხვა მასშტაბი kNN-ს? `StandardScaler` (Recipe 4.2)
2. **მონაცემთა გაწმენდა:** გამოტოვებული მნიშვნელობები (`SimpleImputer`, Recipe 3.11) და კატეგორიული კოდირება (`OneHotEncoder`, Recipe 5.1)
3. **დაზეპირების საფრთხე:** Train/Test Split და მონაცემთა გაჟონვა (Data Leakage)
4. **ჰიპერპარამეტრის შერჩევა:** $k$-Fold Cross-Validation და ვალიდაციის მრუდი (Recipe 11.1)
5. **სრული Pipeline:** მასშტაბირებული vs არამასშტაბირებული kNN-ის შედარება (Recipe 12.1)
6. **სიზუსტის მიღმა:** Confusion Matrix, Precision, Recall, F1 და ROC-AUC (Recipe 11.3, 11.5, 11.6)
7. **Exit Ticket:** საკონტროლო კითხვები

### 0. ბიბლიოთეკების იმპორტი და გარემოს შემოწმება
გაუშვით ქვედა უჯრა. თუ Windows Smart App Control გააქტიურებულია, კოდში ჩადებულია თავსებადობის დამცავი მექანიზმი.

In [ ]:
import sys, types
# Windows Smart App Control fallback shim (Python 3.14)
if 'sklearn.svm._liblinear' not in sys.modules:
    try:
        import sklearn.svm._liblinear
    except Exception:
        sys.modules['sklearn.svm._liblinear'] = types.ModuleType('sklearn.svm._liblinear')

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    roc_auc_score,
    classification_report
)

print("ყველა საჭირო ბიბლიოთეკა წარმატებით ჩაიტვირთა!")

## 1. მასშტაბირების კატასტროფა: რატომ კლავს სხვადასხვა მასშტაბი kNN-ს?

გავიხსენოთ I კვირიდან ევკლიდური მანძილი:
$$d(p, q) = \sqrt{\sum_{i=1}^n (p_i - q_i)^2}$$

თუ გვაქვს ორი მახასიათებელი:
* **ასაკი:** 20-დან 65 წლამდე (სხვაობა ~ 20 წელი $\to 20^2 = 400$)
* **ხელფასი:** 20,000-დან 120,000 ლარამდე (სხვაობა ~ 5,000 ლარი $\to 5,000^2 = 25,000,000$)

ევკლიდურ მანძილში ხელფასის კვადრატული სხვაობა ($25,000,000$) ასაკის სხვაობას ($400$) უბრალოდ „ჩაყლაპავს“! kNN-ისთვის ასაკი გახდება უხილავი.

**გამოსავალი — სტანდარტიზაცია (StandardScaler, Recipe 4.2):**
$$z = \frac{x - \mu}{\sigma}$$
ყველა სვეტის საშუალოს ვხდით $0$-ს, ხოლო სტანდარტულ გადახრას — $1$-ს.

In [ ]:
# ნედლი მონაცემები: 5 კლიენტის ასაკი და წლიური შემოსავალი (GEL)
sample_customers = np.array([
    [22.0, 25000.0],
    [25.0, 50000.0],  # კლიენტი A
    [45.0, 55000.0],  # კლიენტი B (ასაკით 20 წლით უფროსი! შემოსავლით თითქმის იგივე)
    [60.0, 110000.0],
    [35.0, 75000.0]
])

# გამოვთვალოთ ნედლი ევკლიდური მანძილი კლიენტ A-სა და კლიენტ B-ს შორის:
raw_diff = sample_customers[1] - sample_customers[2]
raw_dist = np.sqrt(np.sum(raw_diff ** 2))
print(f"ნედლი ევკლიდური მანძილი A-სა და B-ს შორის: {raw_dist:,.2f}")
print(f"ასაკის წილი კვადრატში: {raw_diff[0]**2:.0f} vs შემოსავლის წილი: {raw_diff[1]**2:,.0f}")

### ✍️ დავალება 1 (# TODO 1): გამოიყენეთ StandardScaler (Recipe 4.2)
1. შექმენით `StandardScaler`-ის ობიექტი.
2. გარდაქმენით `sample_customers` მონაცემები `fit_transform`-ით.
3. გამოთვალეთ სტანდარტიზებული ევკლიდური მანძილი კლიენტ A-სა და B-ს შორის.

In [ ]:
# TODO 1: დაასტანდარტეთ მონაცემები StandardScaler-ის გამოყენებით
# scaler = StandardScaler()
# scaled_customers = scaler.fit_transform(sample_customers)

# YOUR CODE HERE:
scaler = None
scaled_customers = None

# შემოწმება:
if scaled_customers is not None:
    scaled_diff = scaled_customers[1] - scaled_customers[2]
    scaled_dist = np.sqrt(np.sum(scaled_diff ** 2))
    print(f"სტანდარტიზებული მანძილი A-სა და B-ს შორის: {scaled_dist:.4f}")
    print(f"კლიენტი A: ასაკი z={scaled_customers[1, 0]:.3f}, შემოსავალი z={scaled_customers[1, 1]:.3f}")
    print(f"კლიენტი B: ასაკი z={scaled_customers[2, 0]:.3f}, შემოსავალი z={scaled_customers[2, 1]:.3f}")

## 2. მონაცემთა გაწმენდა: Imputation (Recipe 3.11) & One-Hot Encoding (Recipe 5.1)

რეალურ მონაცემებში გვაქვს ორი დიდი გამოწვევა:
1. **გამოტოვებული მნიშვნელობები (`np.nan`):** თუ kNN-ს მივაწვდით `NaN`-ს, მივიღებთ შეცდომას. გამოსავალია მათი შევსება (`SimpleImputer` მედიანის სტრატეგიით).
2. **კატეგორიული ცვლადები (ტექსტი):** ქალაქები „თბილისი“, „ბათუმი“, „ქუთაისი“ არ შეიძლება ჩავწეროთ როგორც $1, 2, 3$, რადგან მანძილი ჩათვლის, რომ ქუთაისი 3-ჯერ დიდია თბილისზე! ამიტომ ვიყენებთ `OneHotEncoder`-ს, რომელიც ქმნის ცალკე $0/1$ ორობით სვეტებს.

In [ ]:
# რიცხვითი მონაცემები გამოტოვებული მნიშვნელობებით (np.nan):
X_numeric = np.array([
    [25.0, 45000.0],
    [30.0, np.nan],    # გამოტოვებული შემოსავალი
    [np.nan, 72000.0],  # გამოტოვებული ასაკი
    [45.0, 110000.0],
    [35.0, 60000.0],
    [np.nan, 52000.0]   # გამოტოვებული ასაკი
])

# კატეგორიული მონაცემები (ქალაქი):
cities = np.array([['Tbilisi'], ['Batumi'], ['Tbilisi'], ['Kutaisi'], ['Batumi'], ['Tbilisi']])

print("ნედლი რიცხვითი მონაცემები:")
print(X_numeric)

### ✍️ დავალება 2 (# TODO 2 & # TODO 3): Imputer და OneHotEncoder
1. გამოიყენეთ `SimpleImputer(strategy='median')` გამოტოვებული უჯრების შესავსებად (`# TODO 2`).
2. გამოიყენეთ `OneHotEncoder(sparse_output=False)` ქალაქების ორობით სვეტებად გადასაქცევად (`# TODO 3`).

In [ ]:
# TODO 2: შეავსეთ გამოტოვებული უჯრები SimpleImputer-ით (Recipe 3.11)
# imputer = SimpleImputer(strategy='median')
# X_imputed = imputer.fit_transform(X_numeric)

X_imputed = None  # YOUR CODE HERE

# TODO 3: გარდაქმენით ქალაქები OneHotEncoder-ით (Recipe 5.1)
# encoder = OneHotEncoder(sparse_output=False)
# cities_encoded = encoder.fit_transform(cities)

cities_encoded = None  # YOUR CODE HERE

# შემოწმება:
if X_imputed is not None and cities_encoded is not None:
    X_combined = np.hstack([X_imputed, cities_encoded])
    print("გასუფთავებული და გაერთიანებული მატრიცის ფორმა:", X_combined.shape)
    print("პირველი სტრიქონი:", X_combined[0])

## 3. მოდელის ვალიდაცია: Train/Test Split და k-Fold კროს-ვალიდაცია

**მანქანური სწავლების ოქროს წესი:** არასოდეს გამოსცადოთ მოდელი იმ მონაცემებზე, რომლებზეც გაწვრთენით!
* **გადავარჯიშება (Overfitting):** მოდელმა დაზეპირდა საწვრთნელი მაგალითები ხმაურიანად ($k=1$ kNN-ში). საწვრთნელზე სიზუსტე 100%-ია, გამოცდაზე — დაბალი.
* **ნაკლებ-სწავლება (Underfitting):** მოდელი ზედმეტად პრიმიტიულია ($k$ ძალიან დიდია). სიზუსტე ყველგან დაბალია.

გამოვიყენოთ რეალური სამედიცინო მონაცემები: **Breast Cancer Wisconsin Diagnostic Dataset** (`load_breast_cancer`).
მიზანი: 30 ბიოლოგიური მახასიათებლის მიხედვით სიმსივნის დიაგნოსტიკა (ავთვისებიანი: Malignant vs კეთილთვისებიანი: Benign).

In [ ]:
# მონაცემთა ნაკრების ჩატვირთვა
data = load_breast_cancer()
X, y = data.data, data.target

print(f"სულ ნიმუშები: {X.shape[0]}, მახასიათებლები: {X.shape[1]}")
print(f"კლასები: {data.target_names[0]} (0) = {np.sum(y == 0)}, {data.target_names[1]} (1) = {np.sum(y == 1)}")
print(f"მახასიათებლების მასშტაბი: 'mean area' [{X[:, 3].min():.1f} - {X[:, 3].max():.1f}] vs 'mean smoothness' [{X[:, 4].min():.4f} - {X[:, 4].max():.4f}]")

# მონაცემთა გაყოფა Train და Test ნაკრებებად (Recipe 11.3)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f"საწვრთნელი ნიმუშები: {len(X_train)}, სატესტო ნიმუშები: {len(X_test)}")

### ✍️ დავალება 3 (# TODO 4): ოპტიმალური k-ს შერჩევა k-Fold კროს-ვალიდაციით (Recipe 11.1)
სატესტო ნაკრებს ვინახავთ სეიფში. საწვრთნელ ნაკრებზე ვატარებთ 5-Fold Cross-Validation-ს $k$-ს სხვადასხვა მნიშვნელობებისთვის:
1. გადაუარეთ ციკლით $k \in [1, 3, 5, 7, 9, 11, 15, 21]$-ს.
2. გამოიყენეთ `cross_val_score(knn, X_train_scaled, y_train, cv=5)`.
3. შეინახეთ თითოეული $k$-სთვის საშუალო სიზუსტე.

In [ ]:
# სტანდარტიზაცია საწვრთნელ მონაცემებზე (CV-ის სადემონსტრაციოდ)
scaler_cv = StandardScaler()
X_train_scaled = scaler_cv.fit_transform(X_train)

k_candidates = [1, 3, 5, 7, 9, 11, 15, 21]
cv_scores = []

# TODO 4: გამოთვალეთ cross_val_score თითოეული k-სთვის (Recipe 11.1)
# for k in k_candidates:
#     knn = KNeighborsClassifier(n_neighbors=k)
#     scores = cross_val_score(knn, X_train_scaled, y_train, cv=5, scoring='accuracy')
#     cv_scores.append(np.mean(scores))

# YOUR CODE HERE:

# ვიზუალიზაცია (თუ cv_scores შევსებულია):
if len(cv_scores) == len(k_candidates):
    best_idx = np.argmax(cv_scores)
    best_k = k_candidates[best_idx]
    print(f"საუკეთესო k = {best_k} (კროს-ვალიდაციის სიზუსტე = {cv_scores[best_idx]*100:.2f}%)")
    
    plt.figure(figsize=(8, 4))
    plt.plot(k_candidates, cv_scores, marker='o', color='royalblue', linewidth=2)
    plt.axvline(best_k, color='crimson', linestyle='--', label=f'Best k={best_k}')
    plt.title('Validation Curve: Accuracy vs k (5-Fold CV)')
    plt.xlabel('k (Number of Neighbors)')
    plt.ylabel('Mean CV Accuracy')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

## 4. სრული Scikit-Learn Pipeline გაჟონვის გარეშე (Recipe 12.1)

**მონაცემთა გაჟონვის (Data Leakage) პრობლემა:**  
თუ სატესტო ნაკრებზე გავუშვებთ `fit_transform()`-ს, სატესტო ნაკრების ინფორმაცია გაჟონავს მოდელში.  
`Pipeline` გარანტიას გაძლევთ:
* `pipeline.fit(X_train, y_train)`: `StandardScaler`-ს აწვრთნის **მხოლოდ** საწვრთნელ მონაცემებზე.
* `pipeline.predict(X_test)`: სატესტო მონაცემებს ატრანსფორმირებს საწვრთნელი პარამეტრებით, გაჟონვის გარეშე!

### ✍️ დავალება 4 (# TODO 5): ააგეთ და შეადარეთ Pipeline მასშტაბირების გარეშე და მასშტაბირებით
1. ააგეთ `Pipeline`, რომელიც შეიცავს `('scaler', StandardScaler())`-ს და `('knn', KNeighborsClassifier(n_neighbors=best_k))`-ს (Recipe 12.1).
2. გაწვრთენით `pipeline.fit(X_train, y_train)`-ით.
3. შეადარეთ მისი სატესტო სიზუსტე არამასშტაბირებულ `raw_knn`-თან.

In [ ]:
# არამასშტაბირებული kNN (Baseline)
raw_knn = KNeighborsClassifier(n_neighbors=11)
raw_knn.fit(X_train, y_train)
acc_unscaled = accuracy_score(y_test, raw_knn.predict(X_test))

# TODO 5: შექმენით და გაწვრთენით სრული Pipeline (Recipe 12.1)
# pipeline = Pipeline([
#     ('scaler', StandardScaler()),
#     ('knn', KNeighborsClassifier(n_neighbors=11))
# ])
# pipeline.fit(X_train, y_train)

pipeline = None  # YOUR CODE HERE

# შემოწმება:
if pipeline is not None:
    acc_scaled = accuracy_score(y_test, pipeline.predict(X_test))
    print(f"არამასშტაბირებული kNN სიზუსტე:  {acc_unscaled * 100:.2f}%")
    print(f"მასშტაბირებული Pipeline სიზუსტე: {acc_scaled * 100:.2f}%")
    print(f"სტანდარტიზაციის ეფექტი:         +{(acc_scaled - acc_unscaled) * 100:.2f}%")

## 5. სიზუსტის მიღმა: აღრევის მატრიცა (Confusion Matrix) და მეტრიკები

სამედიცინო დიაგნოსტიკაში კლასი 0 = Malignant (ავთვისებიანი სიმსივნე), კლასი 1 = Benign (ჯანმრთელი/კეთილთვისებიანი).
გადავაქციოთ ავთვისებიანი სიმსივნე დადებით კლასად ($1$), რათა ოთხოთახიანი ბინა სწორად წავიკითხოთ:
* **TP (True Positive):** მოდელმა ავადმყოფი სწორად ამოიცნო (სიცოცხლე გადარჩა!).
* **TN (True Negative):** ჯანმრთელი სწორად გამოაცხადა ჯანმრთელად.
* **FP (False Positive / Type I Error):** ჯანმრთელს უთხრა ავად ხარო (ცრუ განგაში, ზედმეტი ანალიზი).
* **FN (False Negative / Type II Error):** ავადმყოფს უთხრა ჯანმრთელი ხარო (სასიკვდილო შეცდომა!).

**მეტრიკები (Recipe 11.3):**
* $\text{Precision} = \frac{TP}{TP + FP}$ — სიფრთხილე
* $\text{Recall} = \frac{TP}{TP + FN}$ — დაჭერის უნარი (სამედიცინო ნომერ პირველი მეტრიკა!)
* $F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$ — ჰარმონიული საშუალო

### ✍️ დავალება 5 (# TODO 6): აღრევის მატრიცა და ROC-AUC
1. გამოთვალეთ `confusion_matrix`, `precision_score`, `recall_score`, `f1_score` (Recipe 11.3 & 11.6).
2. გამოთვალეთ `roc_auc_score` და გამოსახეთ ROC მრუდი `roc_curve`-ით (Recipe 11.5).

In [ ]:
# ავთვისებიანი სიმსივნის (Malignant=0) მონიშვნა სამიზნე კლასად (1)
y_test_mal = (y_test == 0).astype(int)
y_pred_mal = (pipeline.predict(X_test) == 0).astype(int) if pipeline else np.zeros_like(y_test)
prob_mal = pipeline.predict_proba(X_test)[:, 0] if pipeline else np.zeros(len(y_test))

# TODO 6: გამოთვალეთ მეტრიკები და აღრევის მატრიცა
# cm = confusion_matrix(y_test_mal, y_pred_mal)
# prec = precision_score(y_test_mal, y_pred_mal)
# rec = recall_score(y_test_mal, y_pred_mal)
# f1 = f1_score(y_test_mal, y_pred_mal)
# auc = roc_auc_score(y_test_mal, prob_mal)

cm = None
prec, rec, f1, auc = None, None, None, None

# YOUR CODE HERE:

if cm is not None:
    tn, fp, fn, tp = cm.ravel()
    print(f"Confusion Matrix:")
    print(f"  TN={tn} (ჯანმრთელი სწორად) | FP={fp} (ცრუ განგაში)")
    print(f"  FN={fn} (გამოტოვებული!)   | TP={tp} (გადარჩენილი!)")
    print(f"Precision: {prec * 100:.2f}%")
    print(f"Recall:    {rec * 100:.2f}%")
    print(f"F1-Score:  {f1 * 100:.2f}%")
    print(f"ROC-AUC:   {auc:.4f}")
    
    # ROC მრუდის აგება
    fpr, tpr, _ = roc_curve(y_test_mal, prob_mal)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {auc:.3f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--', label='Random Guess (AUC=0.5)')
    plt.xlabel('False Positive Rate (ცრუ განგაში)')
    plt.ylabel('True Positive Rate (Recall / მგრძნობელობა)')
    plt.title('Receiver Operating Characteristic (ROC)')
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.show()

## 6. გადაწყვეტილების ზღურბლის (Threshold) მართვა პაციენტის გადასარჩენად

სტანდარტულად ალგორითმი იყენებს ზღურბლს $0.5$. თუ ალბათობა $\ge 0.5$, პაციენტს ეუბნება „დაავადებული ხარ“.
რა მოხდება, თუ ზღურბლს დავწევთ $0.2$-მდე?  
თუ სიმსივნის ალბათობა თუნდაც $20\%$-ია, პაციენტს მაინც გადავამოწმებთ. Recall გაიზრდება, რათა არცერთი ადამიანი არ დაიღუპოს!

In [ ]:
# ზღურბლის ექსპერიმენტი:
threshold_custom = 0.20
y_pred_custom = (prob_mal >= threshold_custom).astype(int)

cm_custom = confusion_matrix(y_test_mal, y_pred_custom)
tn_c, fp_c, fn_c, tp_c = cm_custom.ravel()
rec_c = recall_score(y_test_mal, y_pred_custom)
prec_c = precision_score(y_test_mal, y_pred_custom)

print(f"შედეგები ზღურბლით {threshold_custom}:")
print(f"  დაჭერილი ავადმყოფები (TP): {tp_c} / {tp_c + fn_c}")
print(f"  გამოტოვებული ავადმყოფები (FN): {fn_c}")
print(f"  Recall:    {rec_c * 100:.2f}%")
print(f"  Precision: {prec_c * 100:.2f}%")
print("დასკვნა: ზღურბლის დაწევამ გაზარდა Recall, რითაც გამოტოვებული პაციენტების რიცხვი მინიმუმამდე დაიყვანა!")

## 7. Exit Ticket (ლექციის შეჯამება)

უპასუხეთ შემდეგ ორ კითხვას კომენტარებში ან რვეულში:
1. რატომ არის მონაცემთა გაჟონვა (Data Leakage) სატესტო ნაკრებზე `fit_transform`-ის გაშვება და როგორ იცავს ამისგან `Pipeline`?
2. რატომ არის სამედიცინო დიაგნოსტიკაში Recall უფრო კრიტიკული, ვიდრე Precision, და რატომ გატყუებთ Accuracy არაბალანსირებულ მონაცემებზე?